# Unique SKU Tracker
A retail data platform ingests a batch array $A$ of $N$ product SKU IDs from raw stream logs. 
Write a function `solution(A)` that processes array $A$ and returns the total count of **distinct** SKU IDs in the dataset.

## Example
Input: $A = [2, 1, 1, 2, 3, 1]$  
Output: `3`  
Explanation: The distinct SKU IDs are `1`, `2`, and `3`.

## Constraints
- $N$ is an integer within the range $[0..100,000]$.
- Each element of array $A$ is an integer within the range $[-1,000,000..1,000,000]$.

In [1]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


def solution(A):
    """
    Counts distinct elements using in-place sorting.
    """
    try:
        # Defensive Input Validation
        if A is None:
            raise ValueError("Input array cannot be None")
        if not isinstance(A, (list, tuple)):
            raise TypeError(f"Expected list or tuple, got {type(A).__name__}")
        
        N = len(A)
        if N == 0:
            logger.info("Empty array provided. Distinct count: 0")
            return 0
        
        # Sort in-place to save memory allocation
        A.sort()
        
        count = 1
        for k in range(1, N):
            if A[k] != A[k - 1]:
                count += 1
                
        logger.info(f"Processed {N} items. Found {count} distinct elements.")
        return count

    except (ValueError, TypeError) as e:
        logger.error(f"Validation error: {e}")
        raise

In [2]:
import time
import unittest


class TestDistinct(unittest.TestCase):

    def check_case(self, A, expected):
        """Run solution, time it, and print a detailed PASS/FAIL report."""
        start = time.perf_counter()
        result = solution(A)
        runtime = time.perf_counter() - start

        print()
        print("=" * 70)
        print(f"Input      : {A}")
        print(f"Output     : {result}")
        print(f"Expected   : {expected}")
        print(f"Runtime    : {runtime:.8f}s")
        print(f"Status     : {'PASS' if result == expected else 'FAIL'}")
        print("=" * 70)

        self.assertEqual(result, expected)

    def test_example(self):
        """Standard array with duplicates."""
        self.check_case([2, 1, 1, 2, 3, 1], 3)

    def test_empty_array(self):
        """Edge Case: Empty list."""
        self.check_case([], 0)

    def test_single_element(self):
        """Edge Case: Single element array."""
        self.check_case([42], 1)

    def test_all_identical(self):
        """All elements are the same."""
        self.check_case([5, 5, 5, 5], 1)

    def test_negative_numbers(self):
        """Array with negative and positive values."""
        self.check_case([-10, -5, -10, 0, 5, -5], 4)

    def test_invalid_type(self):
        """Defensive test: Invalid type input."""
        start = time.perf_counter()

        with self.assertRaises(TypeError):
            solution("invalid_string_input")

        runtime = time.perf_counter() - start

        print()
        print("=" * 70)
        print(f"Input      : 'invalid_string_input'")
        print(f"Expected   : TypeError raised")
        print(f"Runtime    : {runtime:.8f}s")
        print(f"Status     : PASS")
        print("=" * 70)


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(TestDistinct)
    unittest.TextTestRunner(verbosity=2).run(suite)

test_all_identical (__main__.TestDistinct.test_all_identical)
All elements are the same. ... 2026-09-06 09:27:19,318 - INFO - Processed 4 items. Found 1 distinct elements.
ok
test_empty_array (__main__.TestDistinct.test_empty_array)
Edge Case: Empty list. ... 2026-09-06 09:27:19,321 - INFO - Empty array provided. Distinct count: 0
ok
test_example (__main__.TestDistinct.test_example)
Standard array with duplicates. ... 2026-09-06 09:27:19,322 - INFO - Processed 6 items. Found 3 distinct elements.
ok
test_invalid_type (__main__.TestDistinct.test_invalid_type)
Defensive test: Invalid type input. ... 2026-09-06 09:27:19,324 - ERROR - Validation error: Expected list or tuple, got str
ok
test_negative_numbers (__main__.TestDistinct.test_negative_numbers)
Array with negative and positive values. ... 2026-09-06 09:27:19,330 - INFO - Processed 6 items. Found 4 distinct elements.
ok
test_single_element (__main__.TestDistinct.test_single_element)
Edge Case: Single element array. ... 2026-09-06 09


Input      : [5, 5, 5, 5]
Output     : 1
Expected   : 1
Runtime    : 0.00107140s
Status     : PASS

Input      : []
Output     : 0
Expected   : 0
Runtime    : 0.00046180s
Status     : PASS

Input      : [1, 1, 1, 2, 2, 3]
Output     : 3
Expected   : 3
Runtime    : 0.00046590s
Status     : PASS

Input      : 'invalid_string_input'
Expected   : TypeError raised
Runtime    : 0.00313030s
Status     : PASS

Input      : [-10, -10, -5, -5, 0, 5]
Output     : 4
Expected   : 4
Runtime    : 0.00089140s
Status     : PASS

Input      : [42]
Output     : 1
Expected   : 1
Runtime    : 0.00063830s
Status     : PASS


# Peak Metric Triplet Optimizer

A real-time financial data pipeline ingests a batch array $A$ of $N$ factor multipliers representing normalized market return signals.
Write a function `solution(A)` that computes and returns the maximal product achievable from any triplet of distinct signals $(P, Q, R)$ where $0 \le P < Q < R < N$.

## Example

Input: $A = [-3, 1, 2, -2, 5, 6]$

Output: `60`

Explanation: The triplet at indices $(2, 4, 5)$ yields the maximal product: $2 \times 5 \times 6 = 60$.

## Constraints

* $N$ is an integer within the range $[3..100,000]$.
* Each element of array $A$ is an integer within the range $[-1,000..1,000]$.

In [ ]:
import time
import unittest
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def solution(A):
    try:
        # 1. Validation checks
        if A is None:
            raise ValueError("Input array cannot be None")
        if not isinstance(A, (list, tuple)):
            raise TypeError(f"Expected list or tuple, got {type(A).__name__}")
        if  len(A) < 3:
            raise ValueError("Array must contain at least 3 elements")
            
        # 2. Business logic
        A.sort()
        cal_1 = A[-3] * A[-2] * A[-1]
        cal_2 = A[0] * A[1] * A[-1]
        return max(cal_1, cal_2)

    except (TypeError, ValueError) as err:
        logger.error(f"Pipeline error in solution: {err}")
        raise
    
    finally:
        logger.info("Batch execution attempt finished.")

In [47]:
import time 
import unittest 

class TestMaxProductOfThree(unittest.TestCase):

    def check_case(self, A, expected):
        """Run solution, benchmark runtime, and print PASS/FAIL status."""
        start = time.perf_counter()
        result = solution(A)
        runtime = time.perf_counter() - start

        print()
        print("=" * 70)
        print(f"Input       : {A}")
        print(f"Output      : {result}")
        print(f"Expected    : {expected}")
        print(f"Runtime     : {runtime:.8f}s")
        print(f"Status      : {'PASS' if result == expected else 'FAIL'}")
        print("=" * 70)

        self.assertEqual(result, expected)

    def test_example(self):
        """Standard example from problem statement."""
        self.check_case([-3, 1, 2, -2, 5, 6], 60)
    
    def test_two_large_negatives(self):
        """Two large negative numbers produce a dominant positive product."""
        self.check_case([-10, -10, 1, 2], 200)

    def test_all_negative(self):
        """All negative numbers; maximum product is the least negative value."""
        self.check_case([-5, -4, -3, -2, -1], -6)

    def test_minimum_size(self):
        """Edge case: Array with exactly 3 elements."""
        self.check_case([-2, -3, -4], -24)

    def test_len_less_than_three(self):
        """Defensive test: Array with fewer than 3 elements raises ValueError."""
        start = time.perf_counter()

        with self.assertRaises(ValueError):
            solution([1, 2])

        runtime = time.perf_counter() - start

        print()
        print("=" * 70)
        print("Input       : [1, 2]")
        print("Expected    : ValueError raised")
        print(f"Runtime     : {runtime:.8f}s")
        print("Status      : PASS")
        print("=" * 70)

    def test_invalid_type(self):
        """Defensive test: Invalid type input raises TypeError."""
        start = time.perf_counter()

        with self.assertRaises(TypeError):
            solution("invalid_string_input")

        runtime = time.perf_counter() - start

        print()
        print("=" * 70)
        print("Input       : 'invalid_string_input'")
        print("Expected    : TypeError raised")
        print(f"Runtime     : {runtime:.8f}s")
        print("Status      : PASS")
        print("=" * 70)

        
if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(TestMaxProductOfThree)
    unittest.TextTestRunner(verbosity=2).run(suite)



test_all_negative (__main__.TestMaxProductOfThree.test_all_negative)
All negative numbers; maximum product is the least negative value. ... 2026-09-06 11:05:15,475 - INFO - Batch execution attempt finished.
ok
test_example (__main__.TestMaxProductOfThree.test_example)
Standard example from problem statement. ... 2026-09-06 11:05:15,478 - INFO - Batch execution attempt finished.
ok
test_invalid_type (__main__.TestMaxProductOfThree.test_invalid_type)
Defensive test: Invalid type input raises TypeError. ... 2026-09-06 11:05:15,480 - ERROR - Pipeline error in solution: Expected list or tuple, got str
2026-09-06 11:05:15,480 - INFO - Batch execution attempt finished.
ok
test_len_less_than_three (__main__.TestMaxProductOfThree.test_len_less_than_three)
Defensive test: Array with fewer than 3 elements raises ValueError. ... 2026-09-06 11:05:15,482 - ERROR - Pipeline error in solution: Array must contain at least 3 elements
2026-09-06 11:05:15,483 - INFO - Batch execution attempt finished.
ok



Input       : [-5, -4, -3, -2, -1]
Output      : -6
Expected    : -6
Runtime     : 0.00050570s
Status      : PASS

Input       : [-3, -2, 1, 2, 5, 6]
Output      : 60
Expected    : 60
Runtime     : 0.00062620s
Status      : PASS

Input       : 'invalid_string_input'
Expected    : TypeError raised
Runtime     : 0.00176760s
Status      : PASS

Input       : [1, 2]
Expected    : ValueError raised
Runtime     : 0.00113780s
Status      : PASS

Input       : [-4, -3, -2]
Output      : -24
Expected    : -24
Runtime     : 0.00064560s
Status      : PASS

Input       : [-10, -10, 1, 2]
Output      : 200
Expected    : 200
Runtime     : 0.00108700s
Status      : PASS


# Sensor Mesh Triangulation Validator

A distributed IoT monitoring pipeline ingests a batch array $A$ of $N$ signal transmission ranges measured from edge sensor beacons.
Write a function `solution(A)` that determines whether any triplet of distinct signal ranges $(P, Q, R)$ with $0 \le P < Q < R < N$ can form a valid geometric triangle satisfying all three metric inequality conditions:

* $A[P] + A[Q] > A[R]$
* $A[Q] + A[R] > A[P]$
* $A[R] + A[P] > A[Q]$

The function must return `1` if at least one triangular triplet exists in array $A$, and `0` otherwise.

## Example

Input: $A = [10, 2, 5, 1, 8, 20]$

Output: `1`

Explanation: The triplet at indices $(0, 2, 4)$ with values $(10, 5, 8)$ satisfies the triangle inequality ($5 + 8 > 10$, $5 + 10 > 8$, and $8 + 10 > 5$).

## Constraints

* $N$ is an integer within the range $[0..100,000]$.
* Each element of array $A$ is an integer within the range $[-2,147,483,648..2,147,483,647]$.

In [82]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def solution(A):

    try:
        N = len(A)
        if N < 3:
            return 0
        A.sort()
        for i in range(0, N - 2):
            if (A[i]+ A[i+1]) > A[i+2]:
                return 1
        return 0
    except (TypeError, ValueError) as err:
        logger.error(f"Pipeline error in solution: {err}")
        raise
    finally:
        logger.info("Batch execution attempt finished.")


In [91]:
import time
import unittest


class TestTriangle(unittest.TestCase):

    def check_case(self, A, expected):
        """Run solution, benchmark runtime, and print PASS/FAIL status."""
        start = time.perf_counter()
        result = solution(A)
        runtime = time.perf_counter() - start

        print()
        print("=" * 70)
        print(f"Input       : {A}")
        print(f"Output      : {result}")
        print(f"Expected    : {expected}")
        print(f"Runtime     : {runtime:.8f}s")
        print(f"Status      : {'PASS' if result == expected else 'FAIL'}")
        print("=" * 70)

        self.assertEqual(result, expected)

    def test_small_arrays(self):
        """Edge Case: Arrays with fewer than 3 elements."""
        self.check_case([], 0)
        self.check_case([10, 20], 0)

    def test_example(self):
        """Standard valid case."""
        self.check_case([10, 2, 5, 1, 8, 20], 1)

    def test_no_triangle(self):
        """Array with no valid triangular triplet."""
        self.check_case([10, 50, 5, 1], 0)

    def test_non_positive_values(self):
        """Edge Case: Array with negative numbers and zero."""
        self.check_case([-3, -2, -1, 0], 0)

    def test_max_integers(self):
        """Boundary Case: Extreme values near 32-bit signed integer limits."""
        self.check_case([2147483647, 2147483647, 2147483647], 1)


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(TestTriangle)
    unittest.TextTestRunner(verbosity=2).run(suite)

test_example (__main__.TestTriangle.test_example)
Standard valid case. ... 2026-09-06 14:35:23,283 - INFO - Batch execution attempt finished.
ok
test_max_integers (__main__.TestTriangle.test_max_integers)
Boundary Case: Extreme values near 32-bit signed integer limits. ... 2026-09-06 14:35:23,286 - INFO - Batch execution attempt finished.
ok
test_no_triangle (__main__.TestTriangle.test_no_triangle)
Array with no valid triangular triplet. ... 2026-09-06 14:35:23,290 - INFO - Batch execution attempt finished.
ok
test_non_positive_values (__main__.TestTriangle.test_non_positive_values)
Edge Case: Array with negative numbers and zero. ... 2026-09-06 14:35:23,292 - INFO - Batch execution attempt finished.
ok
test_small_arrays (__main__.TestTriangle.test_small_arrays)
Edge Case: Arrays with fewer than 3 elements. ... 2026-09-06 14:35:23,295 - INFO - Batch execution attempt finished.
2026-09-06 14:35:23,296 - INFO - Batch execution attempt finished.
ok

---------------------------------------


Input       : [1, 2, 5, 8, 10, 20]
Output      : 1
Expected    : 1
Runtime     : 0.00088870s
Status      : PASS

Input       : [2147483647, 2147483647, 2147483647]
Output      : 1
Expected    : 1
Runtime     : 0.00112510s
Status      : PASS

Input       : [1, 5, 10, 50]
Output      : 0
Expected    : 0
Runtime     : 0.00069310s
Status      : PASS

Input       : [-3, -2, -1, 0]
Output      : 0
Expected    : 0
Runtime     : 0.00092340s
Status      : PASS

Input       : []
Output      : 0
Expected    : 0
Runtime     : 0.00076450s
Status      : PASS

Input       : [10, 20]
Output      : 0
Expected    : 0
Runtime     : 0.00093040s
Status      : PASS


# Edge Node Transmission Overlap Analyzer

A distributed telemetry pipeline ingests a batch array $A$ of $N$ non-negative integers representing the radial transmission coverage ranges of edge gateway nodes deployed sequentially along a linear pipeline corridor. Each node $J$ (for $0 \le J < N$) is positioned at coordinate $(J, 0)$ with a circular broadcast radius of $A[J]$.

Two distinct nodes $J$ and $K$ ($J \ne K$) have overlapping coverage zones if their broadcast discs share at least one common point (including their boundaries).

Write a function `solution(A)` that computes and returns the total number of unordered pairs of nodes whose coverage areas intersect. If the total number of intersecting pairs exceeds $10,000,000$, the function must return `-1` to signal an overflow threshold breach.

## Example

Input: $A = [1, 5, 2, 1, 4, 0]$

Output: `11`

Explanation: There are $11$ overlapping pairs:

* Node $1$ and Node $4$ overlap with each other, and both overlap with all other nodes (yielding $9$ pairs).
* Node $2$ also overlaps with Node $0$ and Node $3$ ($2$ additional pairs).
* Total intersecting pairs = $9 + 2 = 11$.

## Constraints

* $N$ is an integer within the range $[0..100,000]$.
* Each element of array $A$ is an integer within the range $[0..2,147,483,647]$.

In [92]:
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def solution(A):
    try:
        N = len(A)
        if N < 2:
            return 0

        starts = sorted([i - A[i] for i in range(N)])
        ends = sorted([i + A[i] for i in range(N)])

        intersections = 0
        end_idx = 0

        for i in range(N):
            # Advance end_idx past all discs that finished before this disc started
            while end_idx < N and ends[end_idx] < starts[i]:
                end_idx += 1

            # All started discs minus the ones already finished
            intersections += i - end_idx

            if intersections > 10_000_000:
                return -1

        return intersections

    except (TypeError, ValueError) as err:
        logger.error(f"Execution error in solution: {err}")
        raise
    finally:
        logger.info("Disc intersection calculation attempt finished.")

In [98]:
import time
import unittest


class TestDiscIntersections(unittest.TestCase):

    def check_case(self, A, expected):
        """Run solution, benchmark runtime, and print PASS/FAIL status."""
        start = time.perf_counter()
        result = solution(A)
        runtime = time.perf_counter() - start

        print()
        print("=" * 70)
        print(f"Input length: {len(A)}")
        print(f"Output      : {result}")
        print(f"Expected    : {expected}")
        print(f"Runtime     : {runtime:.8f}s")
        print(f"Status      : {'PASS' if result == expected else 'FAIL'}")
        print("=" * 70)

        self.assertEqual(result, expected)

    def test_example(self):
        """Standard example from the problem specification."""
        self.check_case([1, 5, 2, 1, 4, 0], 11)

    def test_small_arrays(self):
        """Edge Case: Fewer than two discs cannot intersect."""
        self.check_case([], 0)
        self.check_case([5], 0)

    def test_no_intersections(self):
        """Case: Discs with zero radius at distinct coordinates."""
        # Intervals: [0, 0], [1, 1], [2, 2] -> no overlap
        self.check_case([0, 0, 0], 0)

    def test_touching_boundaries(self):
        """Boundary Case: Discs that meet exactly at border points."""
        # Disc 0: [-1, 1], Disc 1: [1, 1], Disc 2: [1, 3]
        # All three discs share coordinate 1, so all 3 pairs overlap.
        self.check_case([1, 0, 1], 3)

    def test_all_overlapping(self):
        """Case: Every pair of discs intersects."""
        # 4 discs all overlapping -> 4 * 3 // 2 = 6 pairs
        self.check_case([10, 10, 10, 10], 6)

    def test_overflow_threshold(self):
        """Constraint Case: Intersections exceed 10,000,000 (must return -1)."""
        # 5,000 discs all covering each other:
        # 5000 * 4999 // 2 = 12,497,500 pairs > 10,000,000
        large_a = [10_000] * 5_000
        self.check_case(large_a, -1)


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(TestDiscIntersections)
    unittest.TextTestRunner(verbosity=2).run(suite)

test_all_overlapping (__main__.TestDiscIntersections.test_all_overlapping)
Case: Every pair of discs intersects. ... 2026-09-06 15:15:17,464 - INFO - Disc intersection calculation attempt finished.
ok
test_example (__main__.TestDiscIntersections.test_example)
Standard example from the problem specification. ... 2026-09-06 15:15:17,467 - INFO - Disc intersection calculation attempt finished.
ok
test_no_intersections (__main__.TestDiscIntersections.test_no_intersections)
Case: Discs with zero radius at distinct coordinates. ... 2026-09-06 15:15:17,471 - INFO - Disc intersection calculation attempt finished.
ok
test_overflow_threshold (__main__.TestDiscIntersections.test_overflow_threshold)
Constraint Case: Intersections exceed 10,000,000 (must return -1). ... 2026-09-06 15:15:17,477 - INFO - Disc intersection calculation attempt finished.
ok
test_small_arrays (__main__.TestDiscIntersections.test_small_arrays)
Edge Case: Fewer than two discs cannot intersect. ... 2026-09-06 15:15:17,480 -


Input length: 4
Output      : 6
Expected    : 6
Runtime     : 0.00118770s
Status      : PASS

Input length: 6
Output      : 11
Expected    : 11
Runtime     : 0.00100050s
Status      : PASS

Input length: 3
Output      : 0
Expected    : 0
Runtime     : 0.00115420s
Status      : PASS

Input length: 5000
Output      : -1
Expected    : -1
Runtime     : 0.00423610s
Status      : PASS

Input length: 0
Output      : 0
Expected    : 0
Runtime     : 0.00060390s
Status      : PASS

Input length: 1
Output      : 0
Expected    : 0
Runtime     : 0.00072000s
Status      : PASS

Input length: 3
Output      : 3
Expected    : 3
Runtime     : 0.00063540s
Status      : PASS
